# METR-LA 15-Minute Traffic Speed Forecasting with DGF

This notebook demonstrates spatio-temporal traffic speed forecasting on the **METR-LA** benchmark (15-minute horizon) using DGF's high-level Simple API.

In [ ]:
import dgf
import numpy as np

## 1. Load Dataset & Inspect Schema

Load the 15-minute METR-LA traffic graph from CNS:
- **`sensors`**: 207 highway loop detectors with speed time series and coordinates.
- **`queries`**: Prediction query nodes asking for future speed at $t + 15\text{ min}$.
- **`sensor_to_sensor`**: Directed road network distance graph.

In [ ]:
graph, schema = dgf.io.fetch_metr_la_graph("metr_la_15m", repo="CNS")
dgf.analyse.print_schema(schema)

## 2. Chronological Splits

Extract the standard 70/10/20 chronological splits and remove the `#split` indicator from the schema so it is not treated as an input feature.

In [ ]:
split = graph.node_sets["queries"].features["#split"]
train_idx = np.where(split == b"train")[0]
valid_idx = np.where(split == b"valid")[0]
test_idx = np.where(split == b"test")[0]

del schema.node_sets["queries"].features["#split"]

print(f"Train queries:      {len(train_idx):,}")
print(f"Validation queries: {len(valid_idx):,}")
print(f"Test queries:       {len(test_idx):,}")

## 3. Train Time-Aware GNN Model

Train a spatio-temporal GNN using DGF's Simple API with `time_aware=True` for causal neighborhood sampling.

In [ ]:
model = dgf.learning.train_node_model(
    graph=graph,
    schema=schema,
    target_nodeset="queries",
    target_column="speed",
    time_aware=True,
    train_seed_nodes=train_idx,
    valid_seed_nodes=valid_idx,
    num_train_steps=30_000,
    batch_size=32,
    verbose=1,
)

## 4. Inspect Model Architecture

In [ ]:
model.describe()

## 5. Evaluate on Test Queries

In [ ]:
# Predict on a test sample and compute metrics
model.evaluate(graph, seed_node_idxs=test_idx)